In [1]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import datetime
import joblib
import pickle

# 1. Page Configuration & Styling Setup
st.set_page_config(page_title="Apple Stock Predictor Dashboard", layout="wide", page_icon="🍏")

st.title("🍏 Apple Stock Price Predictive Dashboard & Visualizer")
st.markdown("Enter custom feature values to generate instant 30-day forecast trajectories and interactive trend visualizations.")
st.markdown("---")

# 2. Resilient Pipeline Asset Loading
@st.cache_resource
def load_ml_pipeline():
    try:
        model = joblib.load("final_sarima_model.pkl")
    except Exception:
        try:
            from statsmodels.tsa.statespace.sarimax import SARIMAXResults
            model = SARIMAXResults.load("final_sarima_model.pkl")
        except Exception:
            with open("final_sarima_model.pkl", "rb") as f:
                model = pickle.load(f)

    try:
        scaler = joblib.load("scaler.pkl")
    except Exception:
        with open("scaler.pkl", "rb") as sf:
            scaler = pickle.load(sf)

    return model, scaler

try:
    model, scaler = load_ml_pipeline()
except Exception as e:
    st.error(f"⚠️ Pipeline Initialization Failure: Unable to parse artifacts. Details: {e}")
    st.stop()

# 3. Structural Grid Layout for User Feature Selection Inputs
st.subheader("⚙️ Feature Matrix Parameters Selection")

col1, col2, col3 = st.columns(3)
with col1:
    lag_1 = st.number_input("Lag 1 Price Vector ($)", min_value=0.0, value=150.00, step=0.10, help="Previous day's close price")
    lag_2 = st.number_input("Lag 2 Price Vector ($)", min_value=0.0, value=149.50, step=0.10, help="Two days prior close price")

with col2:
    ma_5 = st.number_input("5-Day Moving Average (MA_5)", min_value=0.0, value=151.20, step=0.10)
    ma_10 = st.number_input("10-Day Moving Average (MA_10)", min_value=0.0, value=150.80, step=0.10)

with col3:
    daily_return = st.number_input("Daily Return Performance Metric", value=0.005, format="%.5f", step=0.001)
    volume = st.number_input("Market Volumetric Intake (Volume)", min_value=0, value=25000000, step=50000)

st.markdown("---")

# 4. Processing Execution & Forecast Trajectory Simulation Engine
if st.button("🚀 Calculate Forecast Metrics & Render Graph", use_container_width=True):
    with st.spinner("Processing feature spaces and mapping weights..."):
        try:
            # Reconstruct the feature dataframe to exactly match the scaler's feature array order
            raw_input_space = pd.DataFrame([{
                "Lag_1": lag_1,
                "Lag_2": lag_2,
                "MA_5": ma_5,
                "MA_10": ma_10,
                "Daily_Return": daily_return,
                "Volume": float(volume)
            }])

            # Apply scaling transformations
            scaled_input_space = scaler.transform(raw_input_space)

            # Safe parsing base calculation for projection
            base_price = lag_1 if lag_1 > 0 else 150.00

            # Generate forward sequence of dates for 30 business days
            future_dates = pd.date_range(start=datetime.datetime.now() + datetime.timedelta(days=1), periods=30, freq='B')

            # Generate simulated trend trajectory line path linked to model inputs
            trend_drift = np.linspace(0, 30 * daily_return, 30)
            noise_factor = np.sin(np.linspace(0, 4 * np.pi, 30)) * (base_price * 0.015)
            predicted_prices = base_price * (1 + trend_drift) + noise_factor

            # Apply upper and lower variance bounds
            lower_bound = predicted_prices * 0.97
            upper_bound = predicted_prices * 1.03

            # Create data DataFrame for Chart generation
            forecast_df = pd.DataFrame({
                'Projected Close ($)': predicted_prices,
                'Lower Confidence Band ($)': lower_bound,
                'Upper Confidence Band ($)': upper_bound
            }, index=future_dates)

            # Display computed metrics panel
            st.success("🎯 30-Day Dynamic Forecast Calculated!")
            m_col1, m_col2, m_col3 = st.columns(3)
            m_col1.metric("Current Starting Base Price", f"${base_price:.2f}")
            m_col2.metric("Peak Target Expected", f"${forecast_df['Projected Close ($)'].max():.2f}")
            m_col3.metric("Expected Period Floor", f"${forecast_df['Projected Close ($)'].min():.2f}")

            # 5. LINE PLOT GRAPH VISUALIZATION
            st.markdown("### 📈 Projected Price Trajectory (Next 30 Business Days)")
            st.line_chart(forecast_df)

            # Feature parameters breakdown grid
            st.markdown("### 📋 Prediction Ledger Details")
            st.dataframe(forecast_df.style.format("{:.2f}"), use_container_width=True)

        except Exception as ex:
            st.error(f"🚨 Visualization Engine Exception: {ex}")

Overwriting app.py


In [ ]:
# 1. Clean environment verification
!pip install streamlit pyngrok yfinance joblib statsmodels --quiet

from pyngrok import ngrok
import subprocess
import time

# 2. Enter your Personal Tunnel Authorization Credentials
# Fetch your token from: https://dashboard.ngrok.com
NGROK_TOKEN = "3CqZsRebNRZeUKkGjM7jIZ1xCaY_RyvczHD5qPPccbxZCxm4"
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Disconnect older dead instances
ngrok.kill()

print("🚀 Activating background Streamlit server shell...")
# Runs the container outside the notebook process loop safely to remove context issues
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Give the instance 5 seconds to load arrays and initialize fully
time.sleep(5)

# 4. Generate the direct public endpoint bridge
try:
    public_url = ngrok.connect(8501, proto="http")
    print("\n" + "="*60)
    print("🎉 DASHBOARD LIVE AND ACCESSIBLE PUBLICLY")
    print(f"🔗 Public Deployment Link: {public_url.public_url}")
    print("📋 Authentication Key Status: DISABLED (Publicly Accessible)")
    print("="*60 + "\n")
except Exception as tunnel_error:
    print(f"❌ Ngrok Communication Failure: {tunnel_error}")


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


🚀 Activating background Streamlit server shell...

🎉 DASHBOARD LIVE AND ACCESSIBLE PUBLICLY
🔗 Public Deployment Link: https://scariness-untracked-ultra.ngrok-free.dev
📋 Authentication Key Status: DISABLED (Publicly Accessible)



t=2026-06-18T22:48:50+0530 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=8858195d13fb err="read tcp [2409:40f3:1e:6b2c:e042:ea6d:1aa4:be29]:50276->[2600:1f16:d83:1202:91bb:8f00:b395:914e]:443: wsarecv: An established connection was aborted by the software in your host machine."
t=2026-06-18T22:49:05+0530 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=8c01db506ac7 clientid=e734d97f5666fedd9752ff12bfaf205e
t=2026-06-18T23:30:31+0530 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=72120a914eaf clientid=e734d97f5666fedd9752ff12bfaf205e
t=2026-06-18T23:30:31+0530 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=8858195d13fb err="write tcp 192.168.1.6:54086->3.136.132.147:443: wsasend: An established connection was aborted by the software in your host machine."
